# 01 — Data Preparation (CUAD v1)

This notebook handles the **exact structure** of the Atticus/CUAD dataset you placed in `data/raw/`:

```
data/raw/
├── CUAD_v1.json            ← SQuAD-format QA file (primary source, 40 MB)
├── master_clauses.csv      ← One row per contract × clause type
├── label_group_xlsx/       ← 28 Excel files, one per clause category
├── full_contract_txt/      ← 510 raw contract .txt files
└── full_contract_pdf/      ← Same contracts as PDFs (not used)
```

We parse **all three usable sources** and merge them into a single clean training dataset.

> **Kernel:** `Legal AI`  
> Run `uv run python -m ipykernel install --user --name legal-ai --display-name "Legal AI"` once if the kernel is missing.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

RAW_DIR       = ROOT / CFG['dataset']['raw_dir']
PROCESSED_DIR = ROOT / CFG['dataset']['processed_dir']
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# ── verify expected files exist ───────────────────────────────────────────────
expected = {
    'CUAD_v1.json'       : RAW_DIR / 'CUAD_v1.json',
    'master_clauses.csv' : RAW_DIR / 'master_clauses.csv',
    'label_group_xlsx/'  : RAW_DIR / 'label_group_xlsx',
    'full_contract_txt/' : RAW_DIR / 'full_contract_txt',
}
for name, path in expected.items():
    status = '✅' if path.exists() else '❌ MISSING'
    print(f'  {status}  {name}')

if not (RAW_DIR / 'CUAD_v1.json').exists() and not (RAW_DIR / 'master_clauses.csv').exists():
    raise FileNotFoundError(
        'Neither CUAD_v1.json nor master_clauses.csv found in data/raw/.\n'
        'Place your Atticus dataset files there and re-run.'
    )

  ✅  CUAD_v1.json
  ✅  master_clauses.csv
  ✅  label_group_xlsx/
  ✅  full_contract_txt/


---
## Source A — `CUAD_v1.json` (SQuAD-format)

This is the richest source. Structure:
```json
{ "data": [ {
    "title": "ContractName",
    "paragraphs": [ {
        "context": "<full contract text>",
        "qas": [ {
            "question": "Highlight the parts...(Category Name)...",
            "answers": [ {"text": "<extracted clause span>", "answer_start": 0} ]
        } ]
    } ]
} ] }
```
Each QA entry = one clause annotation. We extract:
- **text** = the answer span (the actual clause text)
- **context** = the surrounding contract paragraph  
- **label** = clause category parsed from the question string
- **contract_name** = document title

In [2]:
json_path = RAW_DIR / 'CUAD_v1.json'
rows_json = []

if json_path.exists():
    print(f'Loading {json_path.name} ({json_path.stat().st_size / 1e6:.1f} MB) ...')
    with open(json_path, encoding='utf-8') as f:
        cuad = json.load(f)

    # The question text looks like:
    #   "Highlight the parts (if any) of this contract related to \"Governing Law\""
    # We extract the category name from quotes.
    CAT_RE = re.compile(r'\"([^\"]+)\"')

    for doc in cuad['data']:
        contract_name = doc['title']
        for para in doc['paragraphs']:
            context = para['context']  # full contract text for this chunk
            for qa in para['qas']:
                # Parse category from question
                m = CAT_RE.search(qa['question'])
                if not m:
                    continue
                category = m.group(1)

                answers = qa.get('answers', [])
                if answers:
                    # Positive example: clause is present
                    for ans in answers:
                        span = ans['text'].strip()
                        if len(span) > 15:   # skip trivially short spans
                            rows_json.append({
                                'text'         : span,
                                'context'      : context[:1000],  # truncate for memory
                                'label'        : category,
                                'contract_name': contract_name,
                                'source'       : 'json_answer',
                            })
                else:
                    # Negative example: this clause type is NOT present
                    # Useful for training — use a 200-char context snippet as text
                    snippet = context[:200].strip()
                    if len(snippet) > 15:
                        rows_json.append({
                            'text'         : snippet,
                            'context'      : context[:1000],
                            'label'        : f'No {category}',  # negative class
                            'contract_name': contract_name,
                            'source'       : 'json_negative',
                        })

    df_json = pd.DataFrame(rows_json)
    print(f'JSON → {len(df_json):,} rows')
    print(f'  Positive examples : {(df_json["source"]=="json_answer").sum():,}')
    print(f'  Negative examples : {(df_json["source"]=="json_negative").sum():,}')
    print(f'  Unique categories : {df_json["label"].nunique()}')
    display(df_json.head(3))
else:
    df_json = pd.DataFrame()
    print('CUAD_v1.json not found, skipping.')

Loading CUAD_v1.json (40.1 MB) ...
JSON → 26,462 rows
  Positive examples : 12,254
  Negative examples : 14,208
  Unique categories : 81


,text,context,label,contract_name,source
0,DISTRIBUTOR AGREEMENT,EXHIBIT 10.6\n\n ...,Document Name,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer
1,Electric City Corp.,EXHIBIT 10.6\n\n ...,Parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer
2,Electric City of Illinois L.L.C.,EXHIBIT 10.6\n\n ...,Parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer


---
## Source B — `master_clauses.csv`

Wide-format CSV: one row per contract, one column per clause type.  
Each cell contains the extracted clause text (or is empty if the clause isn't present).  
We **melt** it into long format: `contract_name | label | text`.

In [3]:
csv_path = RAW_DIR / 'master_clauses.csv'
df_csv = pd.DataFrame()

if csv_path.exists():
    print(f'Loading {csv_path.name} ...')
    raw_csv = pd.read_csv(csv_path, on_bad_lines='skip', low_memory=False)
    print(f'Shape: {raw_csv.shape}')
    print(f'Columns: {list(raw_csv.columns[:10])} ...')
    display(raw_csv.head(2))
else:
    print('master_clauses.csv not found, skipping.')

Loading master_clauses.csv ...
Shape: (510, 83)
Columns: ['Filename', 'Document Name', 'Document Name-Answer', 'Parties', 'Parties-Answer', 'Agreement Date', 'Agreement Date-Answer', 'Effective Date', 'Effective Date-Answer', 'Expiration Date'] ...


,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/14,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/06,"['July 11 , 2006']",7/11/06,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No


In [4]:
if not raw_csv.empty if 'raw_csv' in dir() else False:
    cols = list(raw_csv.columns)

    # The first column is always the contract/document name
    id_col = cols[0]
    clause_cols = cols[1:]   # all other columns are clause categories

    melted = raw_csv.melt(
        id_vars=[id_col],
        value_vars=clause_cols,
        var_name='label',
        value_name='text',
    ).rename(columns={id_col: 'contract_name'})

    # Drop rows where no clause text was found
    melted = melted[melted['text'].notna() & (melted['text'].astype(str).str.strip() != '')].copy()
    melted['text']   = melted['text'].astype(str).str.strip()
    melted['source'] = 'master_csv'

    df_csv = melted[['text', 'label', 'contract_name', 'source']].copy()
    print(f'master_clauses.csv → {len(df_csv):,} clause rows after melt')
    print(f'Unique categories  : {df_csv["label"].nunique()}')
    display(df_csv.head(3))

master_clauses.csv → 40,610 clause rows after melt
Unique categories  : 82


,text,label,contract_name,source
0,['MARKETING AFFILIATE AGREEMENT'],Document Name,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,master_csv
1,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],Document Name,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,master_csv
2,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],Document Name,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,master_csv


---
## Source C — `label_group_xlsx/` (28 Excel files)

One `.xlsx` per clause category. Each file has columns for the contract name and the extracted clause text.

In [5]:
xlsx_dir = RAW_DIR / 'label_group_xlsx'
df_xlsx  = pd.DataFrame()

if xlsx_dir.exists():
    xlsx_files = sorted(xlsx_dir.glob('*.xlsx'))
    print(f'Found {len(xlsx_files)} xlsx files')

    rows_xlsx = []
    for xf in xlsx_files:
        try:
            xdf = pd.read_excel(xf, engine='openpyxl')
        except Exception as e:
            print(f'  SKIPPED {xf.name}: {e}')
            continue

        # Category name is the file stem (e.g. 'Governing_Law' → 'Governing Law')
        category = xf.stem.replace('_', ' ').replace('-', ' ').strip()

        cols_lower = {c.lower().strip(): c for c in xdf.columns}

        # Find the text column
        text_col = None
        for candidate in ['clause text', 'text', 'answer', 'excerpt', 'provision', 'clause']:
            if candidate in cols_lower:
                text_col = cols_lower[candidate]
                break
        if text_col is None:
            # Fall back: use last column
            text_col = xdf.columns[-1]

        # Find contract name column
        name_col = None
        for candidate in ['document name', 'filename', 'contract', 'file', 'contract name']:
            if candidate in cols_lower:
                name_col = cols_lower[candidate]
                break

        for _, row in xdf.iterrows():
            txt = str(row[text_col]).strip()
            if len(txt) > 15 and txt.lower() not in ('nan', 'none', ''):
                rows_xlsx.append({
                    'text'         : txt,
                    'label'        : category,
                    'contract_name': str(row[name_col]).strip() if name_col else '',
                    'source'       : 'xlsx',
                })

        print(f'  {xf.name}: {len(xdf)} rows → category "{category}"')

    df_xlsx = pd.DataFrame(rows_xlsx)
    print(f'\nXLSX total → {len(df_xlsx):,} rows, {df_xlsx["label"].nunique()} categories')
else:
    print('label_group_xlsx/ not found, skipping.')

Found 28 xlsx files
  Label Report - Anti-assignment, CIC (Group 3).xlsx: 376 rows → category "Label Report   Anti assignment, CIC (Group 3)"
  Label Report - Audit Rights.xlsx: 271 rows → category "Label Report   Audit Rights"
  Label Report - Covenant not to Sue_ Release of Claims.xlsx: 100 rows → category "Label Report   Covenant not to Sue  Release of Claims"
  Label Report - Dates (Group 1).xlsx: 501 rows → category "Label Report   Dates (Group 1)"
  Label Report - Document Name.xlsx: 510 rows → category "Label Report   Document Name"
  Label Report - Governing Law.xlsx: 510 rows → category "Label Report   Governing Law"
  Label Report - Insurance.xlsx: 167 rows → category "Label Report   Insurance"
  Label Report - IP Ownership Assignment.xlsx: 124 rows → category "Label Report   IP Ownership Assignment"
  Label Report - Joint IP Ownership.xlsx: 46 rows → category "Label Report   Joint IP Ownership"
  Label Report - Licenses (Group 4).xlsx: 261 rows → category "Label Report   Lic

---
## Merge all sources

Stack JSON + CSV + XLSX, then deduplicate on clause text.

In [6]:
frames = []
for name, df in [('JSON', df_json), ('CSV', df_csv), ('XLSX', df_xlsx)]:
    if not df.empty:
        print(f'  {name}: {len(df):,} rows')
        frames.append(df[['text', 'label', 'contract_name', 'source']])

if not frames:
    raise RuntimeError('No data loaded from any source — check data/raw/ contents.')

combined = pd.concat(frames, ignore_index=True)
print(f'\nCombined: {len(combined):,} rows')

  JSON: 26,462 rows
  CSV: 40,610 rows
  XLSX: 4,063 rows

Combined: 71,135 rows


---
## Clean, normalise labels, build label map

In [7]:
# Official CUAD 41-category list
CLAUSE_CATEGORIES = [
    'Document Name', 'Parties', 'Agreement Date', 'Effective Date', 'Expiration Date',
    'Renewal Term', 'Notice Period To Terminate Renewal', 'Governing Law',
    'Most Favored Nation', 'Non-Compete', 'Exclusivity', 'No-Solicit Of Customers',
    'No-Solicit Of Employees', 'Non-Disparagement', 'Termination For Convenience',
    'ROFR/ROFO/ROFN', 'Change Of Control', 'Anti-Assignment', 'Revenue/Profit Sharing',
    'Price Restrictions', 'Minimum Commitment', 'Volume Restriction',
    'IP Ownership Assignment', 'Joint IP Ownership', 'License Grant',
    'Non-Transferable License', 'Affiliate License-Licensor', 'Affiliate License-Licensee',
    'Unlimited/All-You-Can-Eat-License', 'Irrevocable Or Perpetual License',
    'Source Code Escrow', 'Post-Termination Services', 'Audit Rights',
    'Uncapped Liability', 'Cap On Liability', 'Liquidated Damages',
    'Warranty Duration', 'Insurance', 'Covenant Not To Sue',
    'Third Party Beneficiary', 'Arbitration',
]
# Negative-class versions (from CUAD_v1.json unanswered QAs)
NEG_CATEGORIES = [f'No {c}' for c in CLAUSE_CATEGORIES]
ALL_CATEGORIES  = CLAUSE_CATEGORIES + NEG_CATEGORIES

def clean_text(t):
    if not isinstance(t, str): return ''
    t = re.sub(r'\s+', ' ', t).strip().replace('\x00', '')
    return t

def normalise_label(lbl: str) -> str:
    """Fuzzy-match a raw label string to the official CUAD category list."""
    lbl = lbl.strip()
    if lbl in ALL_CATEGORIES:
        return lbl
    lbl_l = lbl.lower()
    for cat in ALL_CATEGORIES:
        if cat.lower() == lbl_l:
            return cat
    for cat in ALL_CATEGORIES:
        if cat.lower().replace(' ', '') == lbl_l.replace(' ', '').replace('_', ''):
            return cat
    for cat in CLAUSE_CATEGORIES:
        if cat.lower() in lbl_l or lbl_l in cat.lower():
            return cat
    return 'Other'

combined['text']  = combined['text'].apply(clean_text)
combined['label'] = combined['label'].astype(str).apply(normalise_label)

# Drop noise
combined = combined[combined['text'].str.len() > 15].copy()
combined = combined[combined['label'] != 'nan'].copy()

# Deduplicate: keep first occurrence of each unique clause text
before = len(combined)
combined = combined.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'Dedup: {before:,} → {len(combined):,} rows')

# Label map
all_labels = sorted(combined['label'].unique())
label2id   = {lbl: i for i, lbl in enumerate(all_labels)}
id2label   = {i: lbl for lbl, i in label2id.items()}
combined['label_id'] = combined['label'].map(label2id)

print(f'Final: {len(combined):,} samples, {len(label2id)} categories')
display(combined.head(5))

Dedup: 38,329 → 20,744 rows
Final: 20,744 samples, 52 categories


,text,label,contract_name,source,label_id
0,DISTRIBUTOR AGREEMENT,Document Name,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer,8
1,Electric City Corp.,Parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer,39
2,Electric City of Illinois L.L.C.,Parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer,39
3,Electric City of Illinois LLC,Parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer,39
4,"7th day of September, 1999.",Agreement Date,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,json_answer,2


---
## Dataset statistics

In [8]:
print(f'=== Dataset Summary ===')
print(f'Total samples     : {len(combined):,}')
print(f'Unique categories : {combined["label"].nunique()}')
print(f'Unique contracts  : {combined["contract_name"].nunique():,}')
print(f'Sources           :')
print(combined['source'].value_counts().to_string())

print(f'\nLabel distribution:')
dist = combined['label'].value_counts()
# Show positives and negatives separately
pos = dist[~dist.index.str.startswith('No ')]
neg = dist[dist.index.str.startswith('No ')]
print(f'  Positive (clause present): {pos.sum():,} rows across {len(pos)} categories')
print(f'  Negative (clause absent) : {neg.sum():,} rows across {len(neg)} categories')
print()
print(dist.head(20).to_string())

=== Dataset Summary ===
Total samples     : 20,744
Unique categories : 52
Unique contracts  : 1,327
Sources           :
source
json_answer      10525
master_csv        6396
xlsx              3315
json_negative      508

Label distribution:
  Positive (clause present): 20,236 rows across 41 categories
  Negative (clause absent) : 508 rows across 11 categories

label
Parties                        2287
Governing Law                  1334
Other                          1144
Audit Rights                   1061
Anti-Assignment                 971
License Grant                   903
Insurance                       880
Document Name                   861
Expiration Date                 794
Cap On Liability                774
Minimum Commitment              722
Agreement Date                  686
Exclusivity                     606
Post-Termination Services       587
Revenue/Profit Sharing          579
Termination For Convenience     574
IP Ownership Assignment         556
Uncapped Liability  

In [9]:
# Text length stats
combined['word_count'] = combined['text'].str.split().str.len()
print('Word count stats:')
print(combined['word_count'].describe().round(1).to_string())

pct_long = (combined['word_count'] > 350).mean() * 100
print(f'\n{pct_long:.1f}% of clauses exceed ~350 words (will be truncated at 512 tokens)')

Word count stats:
count    20744.0
mean        67.1
std        105.2
min          1.0
25%         19.0
50%         38.0
75%         75.0
max       2612.0

2.3% of clauses exceed ~350 words (will be truncated at 512 tokens)


---
## Save

In [10]:
save_cols = ['text', 'label', 'label_id', 'contract_name', 'source']
combined[save_cols].to_csv(PROCESSED_DIR / 'contracts_clean.csv', index=False)

with open(PROCESSED_DIR / 'label2id.json', 'w') as f:
    json.dump(label2id, f, indent=2)
with open(PROCESSED_DIR / 'id2label.json', 'w') as f:
    json.dump(id2label, f, indent=2)

# Also save dataset stats for reference
stats = {
    'total_samples'    : len(combined),
    'num_categories'   : len(label2id),
    'unique_contracts' : int(combined['contract_name'].nunique()),
    'label_counts'     : combined['label'].value_counts().to_dict(),
}
with open(PROCESSED_DIR / 'dataset_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print('Saved:')
print(f'  contracts_clean.csv  ({len(combined):,} rows)')
print(f'  label2id.json        ({len(label2id)} labels)')
print(f'  dataset_stats.json')
print('\n✅ Ready for notebook 02_finetune.ipynb')

Saved:
  contracts_clean.csv  (20,744 rows)
  label2id.json        (52 labels)
  dataset_stats.json

✅ Ready for notebook 02_finetune.ipynb
